# NB10 — Methodenvergleich: Avg, Cert, MaxMin(c), Fair(alpha, eps)

Vergleicht vier Koeffizientenmethoden gegen die Baseline `lambda = p` innerhalb der festen
Interpolationsfamilie `theta(lambda) = theta_0 + D lambda`.

**Dieses Notebook definiert KEINE Methode selbst.** Alle Abbildungen kommen aus
`src/coefficient_portfolio.py`, Merge-Arithmetik aus `src/merge.py`, Reward-Cache und
Metrik aus `src/proxy_validation.py`, lambda-Schluessel aus `src/lambda_utils.py`.
Der Gate G6 in `verify_rs_ppo_setup.py` durchsucht auch `.ipynb`-Dateien nach
Doppeldefinitionen; eine Inline-Kopie waere ein echter Verstoss, nicht nur Stilfrage.

**Zweiphasiger Aufbau:**

| Phase | Was | ArmoRM? | Wiederholbar? |
|---|---|---|---|
| **A** | `lambda = f(p, R)`, Bewegungsdiagnostik, Budgetplanung | nein | ja, beliebig oft |
| **GATE** | Vorregistrierung schreiben und einfrieren | nein | einmalig |
| **B** | `U_p` an allen `lambda` messen, Ergebnistabelle | **ja** | **einmalig** |

Phase A kostet Sekunden. Phase B ist einmalig: jeder ArmoRM-Kontakt ohne vorherige
Vorregistrierung verbrennt die Auswertung. Das Gate bricht ohne eingefrorene, zur
lambda-Tabelle passende Vorregistrierung ab.

**Endtabelle:** `p | Methode | lambda | U_p(p) | U_p(lambda) | Delta U_p | ||lambda-p||_2 | ||lambda-p||_R`

---

**Metrik (Lingxiao freigegeben):** primaer ist die **rohe** praeferenzgewichtete Summe
`U_p(lambda) = sum_i p_i r_i(theta(lambda))`, realisiert als Identitaets-Deklaration
`r~_i := r_i` — `preference_utility(..., normalization="identity")`. Rang und Min-Max
laufen als Robustheitsvergleich aus derselben Reward-Matrix mit, ohne zusaetzliche
Auswertungen. NB06.1 rechnete auf `rank-U_p`; diese Metrik ist ersetzt.

**Cert ist eine Zertifikatszeile, keine Messzeile.** NB09.1 `run1` hat `t* = 0` fuer 77/77
Praeferenzen auf beiden Matrizen bewiesen, Dualzertifikat `R^-1 1 > 0`, gegen die
LP-Formulierung 4000/4000 geprueft. Cert gibt daher per Theorem `lambda = p`, und
`U_p(lambda_Cert) = U_p(p)` gilt **exakt**. Cert erzeugt keine zusaetzliche ArmoRM-Auswertung.

**Zwei Eigenschaften, die beim Berichten nicht untergehen duerfen:**

1. **MaxMin(c) ist an Simplex-Vertices fuer jedes `c < 1` LEER.** Dort zeigt
   `d_0 = Pi_0(R p)` nach aussen, bewegen laesst sich nur nach innen; der Minimalabstand
   ist exakt `||d_0||_R`. Das Portfolio meldet `INFEASIBLE_BALL_MISSES_SIMPLEX` statt
   still `p` zurueckzugeben — eine leere zulaessige Menge ist kein Kollaps.
2. **Fair(alpha, eps) ist praeferenz-blind.** Sein Ziel `U_alpha(R v + eps)` haengt nur von
   der Verschiebung ab; `p` tritt ausschliesslich ueber die Simplex-Schranken ein. Fuer alle
   inneren Praeferenzen ist `v` identisch. Das passt zur Rolle als orthogonale
   Fairness-Achse, heisst aber: Fair ist **keine** Methode der Form `lambda = f(p, R)` im
   Sinne von RQ1 und muss so beschrieben werden.

## 1. Repository

In [ ]:
%cd /content
import os, shutil

repo_path = "/content/master-thesis"
repo_url = "https://github.com/NZhang137/master-thesis.git"

if os.path.isdir(os.path.join(repo_path, ".git")):
    %cd /content/master-thesis
    !git pull
else:
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)
    !git clone {repo_url} {repo_path}
    %cd /content/master-thesis

## 2. Laufzeit und Abhaengigkeiten

In [ ]:
!nvidia-smi || echo "Keine GPU — Phase A laeuft trotzdem."

In [ ]:
# Phase A braucht nur numpy/scipy/pandas/pyyaml.
!pip install -q -U "pandas==2.2.2" "numpy<2.1" scipy pyyaml
# Phase B zusaetzlich (erst entkommentieren, wenn das Gate offen ist):
# !pip install -q -U transformers peft accelerate bitsandbytes safetensors

## 3. Einstellungen

In [ ]:
from __future__ import annotations

import gc, json, hashlib, sys, zipfile
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path("/content/master-thesis").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RUN_TAG = "nb10_run1"

# --- Eingaben (identisch zu NB06.1, damit die Kette nicht reisst) ------------
CONFIG_PATH       = PROJECT_ROOT / "configs/tinyllama_helpsteer2_armorm.yaml"
COSINE_MATRIX_CSV = PROJECT_ROOT / "results/tinyllama_helpsteer2_R/tinyllama_helpsteer2_R_cos.csv"
GRAM_MATRIX_CSV   = PROJECT_ROOT / "results/tinyllama_helpsteer2_R/tinyllama_helpsteer2_R_gram.csv"

# --- Ausgaben ---------------------------------------------------------------
RESULTS_DIR  = PROJECT_ROOT / "results" / f"nb10_method_comparison_{RUN_TAG}"
LAMBDA_CSV   = RESULTS_DIR / "lambda_table.csv"
PREREG_JSON  = RESULTS_DIR / "nb10_preregistration.json"
REWARD_CACHE = RESULTS_DIR / "reward_cache.jsonl"
FINAL_CSV    = RESULTS_DIR / "method_comparison.csv"
ROBUST_CSV   = RESULTS_DIR / "normalization_robustness.csv"
REPORT_JSON  = RESULTS_DIR / "nb10_report.json"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# NB09.1 run1: matrices_agree = True, primary_matrix = R_cos.
PRIMARY_MATRIX = "R_cos"

# --- Methodenparameter (Phase A frei wiederholbar; VOR dem Gate einfrieren) --
RHO_AVG    = 0.5                                # NB06.1: RHO_M1_PENALTY
CERT_C     = 0.5                                # Trust-Region von Cert
CERT_EPS   = 1e-8
C_GRID     = [0.0, 0.25, 0.5, 0.75, 0.9, 1.0]   # MaxMin: 0 aggressiv, >=1 Zertifikat
ALPHA_GRID = [0.0, 1.0, 2.0]                    # Fair: 0 utilitaristisch, 1 Nash
EPS_GRID   = [0.02, 0.05, 0.10]                 # Fair: Floor-Relaxation

# --- Phase B ----------------------------------------------------------------
RUN_REWARD_COLLECTION = False   # bleibt False, bis das Gate offen ist
REWARD_PROMPT_PATH    = RESULTS_DIR / "nb10_reward_prompts.jsonl"
MAX_NEW_TOKENS        = 256
REPETITION_PENALTY    = 1.15
NO_REPEAT_NGRAM_SIZE  = 5
LAMBDA_DEDUP_DECIMALS = 8       # identisch zu src.lambda_utils.lambda_key

print(f"RUN_TAG         = {RUN_TAG}")
print(f"Ergebnisordner  = {RESULTS_DIR}")
print(f"Primaere Matrix = {PRIMARY_MATRIX}")
print(f"Reward-Phase    = {'AKTIV' if RUN_REWARD_COLLECTION else 'aus (nur Phase A)'}")

## 4. Importe aus `src/`

Was hier importiert wird, wird hier **nicht** neu definiert. Die Methoden sind der
Gegenstand der Arbeit; eine zweite Implementierung im Notebook koennte von der
getesteten abweichen, ohne dass es jemand merkt.

In [ ]:
from src.experiment_config import get_attribute_order, load_experiment_config, validate_preference_vectors
from src.proxy_validation import (
    build_search_set,
    coefficient_key,
    collect_reward_matrix,
    load_labeled_matrix_csv,
    normalization_agreement,
    preference_utility,
    write_json,
)
from src.coefficient_portfolio import (
    avg,                 # = m1_plus, Nomenklatur v13
    cert,                # = c1_plus_plus
    fair_alpha_eps,
    floor_lp_at_p,
    improvements,
    maxmin_c,
    maxmin_center,
)
from src.lambda_utils import lambda_key

print("Portfolio importiert. Keine Methode wird in diesem Notebook definiert.")

## 5. Konfiguration, R-Matrizen und Praeferenzen

In [ ]:
config      = load_experiment_config(CONFIG_PATH)
ATTRIBUTES  = get_attribute_order(config)
PREFERENCES = validate_preference_vectors(config)
m           = len(ATTRIBUTES)

# Die Matrizen werden in Abschnitt 5a+ regime-abhaengig aufgeloest und geladen. Hier
# NICHT vorab laden: die Pfadkonstanten aus Abschnitt 2 weisen kein Regime aus.
R_cos = R_gram = R = None
eigenvalues = None


### 5a+. Provenienz — Regime, Adapter, Matrizen

Der teuerste stille Fehler dieses Notebooks waere, ein Adapterregime zu mergen und mit der
Geometrie eines anderen zu korrigieren. Das Notebook liefe durch und produzierte bedeutungslose
Zahlen. Diese Zelle laesst das scheitern, bevor GPU-Zeit verbraucht wird.


In [ ]:
REGIME = "rs_ppo"          # "rs_ppo" oder "sft" — bestimmt Adapter UND Matrizen

# 8-bit spiegelt exakt den Belohnungspfad des PPO-Laufs (ppo_log: armorm_precision=8bit).
# Das ist die Voraussetzung des Obergrenzen-Arguments: der Pruefer muss derselbe sein wie
# das Trainingssignal. bf16 waere hoeherpraezise, aber dann eine bewusst abweichende
# Evaluation — dann hier "bfloat16" eintragen UND es in Kapitel 6 als Abweichung berichten.
ARMORM_PRECISION = "8bit"

# (1) Die Praeferenzen duerfen nur aus einer Quelle kommen. Config und src/preferences.py
#     sind zwei Kopien; weichen sie ab, misst das Notebook etwas anderes als der Rest
#     des Projekts.
from src.preferences import PREFERENCES as PREFERENCES_MODULE

_config_names, _module_names = set(PREFERENCES), set(PREFERENCES_MODULE)
if _config_names != _module_names:
    raise AssertionError(
        "Praeferenzen weichen ab.\n"
        f"  nur in der Config:          {sorted(_config_names - _module_names)}\n"
        f"  nur in src/preferences.py:  {sorted(_module_names - _config_names)}\n"
        "Eine der beiden Kopien ist veraltet. Angleichen, bevor irgendetwas laeuft."
    )
for _name in sorted(_config_names):
    if not np.allclose(np.asarray(PREFERENCES[_name], float),
                       np.asarray(PREFERENCES_MODULE[_name], float), atol=1e-12):
        raise AssertionError(f"Praeferenz {_name!r} hat in Config und Modul verschiedene Werte.")
print(f"[OK] Praeferenzen konsistent: {len(_config_names)} Eintraege, {sorted(_config_names)}")

# (2) Adapter aufloesen. Die RS-PPO-Laeufe liegen unter den NB08-Pfaden, die SFT-Adapter
#     unter adapters/. Es wird NICHT geraten: passt kein Layout vollstaendig, bricht es ab.
ADAPTER_LAYOUTS = {
    "rs_ppo": [
        "results/rs_ppo_armorm_circular/rs_runs/ppo_{axis}/adapter",
        "results/rs_ppo_armorm_circular/rs_runs/ppo_{axis}",
    ],
    "sft": [
        str(config.get("adapter_dir", "adapters")) + "/tinyllama-helpsteer2-{axis}-adapter",
        "adapters/tinyllama-helpsteer2-{axis}-adapter",
    ],
}

def _resolve_adapters(regime):
    for pattern in ADAPTER_LAYOUTS[regime]:
        paths = {a: (PROJECT_ROOT / pattern.format(axis=a)).resolve() for a in ATTRIBUTES}
        if all((p / "adapter_config.json").is_file() for p in paths.values()):
            return pattern, paths
    tried = "\n".join("  " + p for p in ADAPTER_LAYOUTS[regime])
    raise FileNotFoundError(
        f"Keine vollstaendige Adaptermenge fuer Regime {regime!r} gefunden. Geprueft:\n{tried}\n"
        "Pfad korrigieren, statt das Notebook auf ein anderes Regime ausweichen zu lassen."
    )

ADAPTER_PATTERN, ADAPTER_PATHS = _resolve_adapters(REGIME)
print(f"[OK] Adapter ({REGIME}): {ADAPTER_PATTERN}")
for _a, _p in ADAPTER_PATHS.items():
    print(f"       {_a:12s} {_p.relative_to(PROJECT_ROOT)}")

# (3) Matrizen regime-abhaengig aufloesen. Der Dateiname ist KEIN Nachweis: der
#     bisherige Pfad results/tinyllama_helpsteer2_R/ sagt nicht, aus welchem Lauf die
#     Matrix stammt. Deshalb wird sie hier explizit ueber das Regime gewaehlt und ihr
#     Hash in den Bindungsnachweis aufgenommen.
MATRIX_LAYOUTS = {
    "rs_ppo": [
        ("results/nb09_1_geometry_run1/R_cos.csv",
         "results/nb09_1_geometry_run1/R_gram.csv"),
        ("results/rs_ppo_armorm_circular/geometry/R_cos.csv",
         "results/rs_ppo_armorm_circular/geometry/R_gram.csv"),
        ("results/rs_ppo_armorm_circular/nb09_1_R_cos.csv",
         "results/rs_ppo_armorm_circular/nb09_1_R_gram.csv"),
    ],
    "sft": [
        ("results/tinyllama_helpsteer2_R/tinyllama_helpsteer2_R_cos.csv",
         "results/tinyllama_helpsteer2_R/tinyllama_helpsteer2_R_gram.csv"),
    ],
}

def _resolve_matrices(regime):
    for cos_rel, gram_rel in MATRIX_LAYOUTS[regime]:
        cos_path, gram_path = PROJECT_ROOT / cos_rel, PROJECT_ROOT / gram_rel
        if cos_path.is_file() and gram_path.is_file():
            return cos_path.resolve(), gram_path.resolve()
    tried = "\n".join(f"  {c}\n  {g}" for c, g in MATRIX_LAYOUTS[regime])
    raise FileNotFoundError(
        f"Keine R-Matrizen fuer Regime {regime!r} gefunden. Geprueft:\n{tried}\n"
        "Die aus NB09.1 berechneten RS-PPO-Matrizen dorthin kopieren, statt auf einen "
        "Pfad auszuweichen, dessen Regime der Name nicht ausweist."
    )

COSINE_MATRIX_CSV, GRAM_MATRIX_CSV = _resolve_matrices(REGIME)
R_cos = load_labeled_matrix_csv(COSINE_MATRIX_CSV, ATTRIBUTES)
R_gram = load_labeled_matrix_csv(GRAM_MATRIX_CSV, ATTRIBUTES)
R = R_cos if PRIMARY_MATRIX == "R_cos" else R_gram
_eig = np.linalg.eigvalsh(R)
assert _eig.min() > 0, f"R ist nicht positiv definit (min EW {_eig.min():.3e})."
print(f"[OK] Matrizen ({REGIME}): {COSINE_MATRIX_CSV.relative_to(PROJECT_ROOT)}")
print(f"       dominanter Anteil {_eig.max() / _eig.sum() * 100:.2f} % "
      f"(RS-PPO erwartet ~35.1 %, SFT ~46 %)")

eigenvalues = _eig
offdiag = R[np.triu_indices(m, 1)]
print(f"Attribute: {list(ATTRIBUTES)}")
print(f"Eigenwerte {PRIMARY_MATRIX}: {np.round(eigenvalues, 4)}")
print(f"Off-Diagonalen: {offdiag.min():.4f} .. {offdiag.max():.4f} "
      f"(Mittel {offdiag.mean():.4f})")

# Pi0(R 1) != 0 ist die Bedingung, unter der Fair dem Kollaps entkommt (Prop 26, v13.2).
# Bei perfekt aequikorreliertem R waere sie verletzt und Fair gaebe p zurueck.
escape = R @ np.ones(m)
escape = escape - escape.mean()
print(f"||Pi_0(R 1)|| = {np.linalg.norm(escape):.6f}  "
      f"({'Fair kann entkommen' if np.linalg.norm(escape) > 1e-9 else 'DEGENERIERT: Fair kollabiert'})")

display(pd.DataFrame(R, index=list(ATTRIBUTES), columns=list(ATTRIBUTES)).round(4))


### 5a. Praeferenzmenge

Die Config enthaelt die **11** benannten Praeferenzen: fuenf Vertices (`only_*`), fuenf
dominante und `balanced`. Mit 64 Dirichlet-Punkten ergibt der volle Suchsatz **75** Punkte.

NB09.1 registrierte 13 Praeferenzen und 77 Punkte vor; `quality_focused` und
`detailed_answer` wurden seither entfernt und `uniform` in `balanced` umbenannt. Wo im
Notebook 13 oder 77 steht, ist das eine historische NB09.1-Angabe. Die Abweichung ist in der
Vorregistrierung als solche vermerkt und gehoert so berichtet.

`USE_FULL_SEARCH_SET` schaltet Phase A auf alle 75 Punkte; Phase B bleibt in jedem Fall bei
den 11 benannten.

In [ ]:
# 11 benannte Praeferenzen + 64 Dirichlet-Punkte = 75. Die 77 aus NB09.1 sind
# historisch: sie enthielten quality_focused und detailed_answer, die entfernt wurden.
USE_FULL_SEARCH_SET = True

PREF_SET = [(name, np.asarray(vec, dtype=np.float64)) for name, vec in PREFERENCES.items()]

if USE_FULL_SEARCH_SET:
    seen = {tuple(np.round(v, 9)) for _, v in PREF_SET}
    B = build_search_set(m, n_dirichlet=64, dirichlet_alpha=1.0,
                         preferences=list(PREFERENCES.values()), seed=137)
    for j, vec in enumerate(B):
        key = tuple(np.round(vec, 9))
        if key not in seen:
            seen.add(key)
            PREF_SET.append((f"dirichlet_{j:02d}", np.asarray(vec, dtype=np.float64)))

for name, p in PREF_SET:
    assert abs(p.sum() - 1.0) < 1e-9 and np.all(p >= -1e-12), f"{name} nicht im Simplex."

INTERIOR = [name for name, p in PREF_SET if np.all(p > 1e-12)]
print(f"|P| = {len(PREF_SET)}  (davon {len(INTERIOR)} innere, {len(PREF_SET) - len(INTERIOR)} auf dem Rand)")
display(pd.DataFrame([p for _, p in PREF_SET],
                     index=[n for n, _ in PREF_SET], columns=list(ATTRIBUTES)).round(4))

### 5b. Phase-B-Teilmenge

Phase A ist reward-frei und darf breit laufen. Phase B kostet GPU-Stunden und ArmoRM-Kontakt.
Die Teilmenge ist als REGEL formuliert, nicht als Liste, damit sie nicht nachtraeglich
waehlbar ist.


In [ ]:
PHASE_B_RULE = "all named preferences from the experiment config; no Dirichlet draws"
PREF_SET_B = [(name, p) for name, p in PREF_SET if name in PREFERENCES]
PHASE_B_NAMES = {name for name, _ in PREF_SET_B}

assert len(PREF_SET_B) == len(PREFERENCES), (
    f"PREF_SET_B hat {len(PREF_SET_B)} Eintraege, PREFERENCES {len(PREFERENCES)}."
)
assert PHASE_B_NAMES, "Phase-B-Praeferenzmenge ist leer."
print(f"Phase A: |P| = {len(PREF_SET)}    Phase B: |P_B| = {len(PREF_SET_B)}")
print("Phase B:", sorted(PHASE_B_NAMES))


## 6. Phase A — alle `lambda` berechnen (reward-frei)

Sekunden, keine GPU, kein ArmoRM. Beliebig oft wiederholbar, bis die Gitter stehen.

In [ ]:
def norm_R(v):
    return float(np.sqrt(max(v @ R @ v, 0.0)))


rows = []
for pname, p in PREF_SET:
    base = {"p_name": pname, **{f"p_{a}": float(p[i]) for i, a in enumerate(ATTRIBUTES)}}

    results = []

    lam = avg(p, R, RHO_AVG)
    results.append(("Avg", f"rho={RHO_AVG}", lam, "OK", {}))

    lam_cert, t_cert = cert(p, R, CERT_C, CERT_EPS)
    t_lp, collapsed = floor_lp_at_p(R, p)
    results.append(("Cert", "", lam_cert,
                    "FLOOR_COLLAPSED" if collapsed else "FLOOR_NONTRIVIAL",
                    {"t_star": t_cert, "t_star_lp": t_lp}))

    for c in C_GRID:
        lam_mm, t_mm, status = maxmin_c(p, R, c)
        results.append(("MaxMin", f"c={c}", lam_mm, status, {"t_star": t_mm}))

    for alpha in ALPHA_GRID:
        for eps in EPS_GRID:
            lam_f, u_f, status = fair_alpha_eps(p, R, alpha, eps)
            results.append(("Fair", f"alpha={alpha},eps={eps}", lam_f, status, {"u_alpha": u_f}))

    for method, params, lam, status, extra in results:
        row = dict(base, method=method, params=params, status=status, **extra)
        if lam is None:
            row.update({f"lam_{a}": np.nan for a in ATTRIBUTES})
            row.update({"dist_l2": np.nan, "dist_R": np.nan, "proxy_pRlam": np.nan,
                        "min_delta": np.nan, "moved": False, "usable": False})
        else:
            v = lam - p
            row.update({f"lam_{a}": float(lam[i]) for i, a in enumerate(ATTRIBUTES)})
            row.update({"dist_l2": float(np.linalg.norm(v)),
                        "dist_R": norm_R(v),
                        "proxy_pRlam": float(p @ R @ lam),
                        "min_delta": float(np.min(improvements(p, R, lam))),
                        "moved": bool(np.linalg.norm(v) > 1e-8),
                        "usable": True})
        rows.append(row)

lam_df = pd.DataFrame(rows)
lam_df.to_csv(LAMBDA_CSV, index=False)
lam_cols = [f"lam_{a}" for a in ATTRIBUTES]
p_cols = [f"p_{a}" for a in ATTRIBUTES]
print(f"{len(lam_df)} lambda-Zeilen -> {LAMBDA_CSV}")
display(lam_df.head(20))

### 6a. Bewegungsdiagnostik

Kollaps und leere zulaessige Menge sind verschiedene Dinge und werden getrennt gezaehlt.

In [ ]:
key_series = lam_df["method"] + lam_df["params"].map(lambda s: f"({s})" if s else "")
summary = (lam_df.assign(key=key_series).groupby("key")
           .agg(n=("moved", "size"),
                n_usable=("usable", "sum"),
                n_moved=("moved", "sum"),
                mean_dist_l2=("dist_l2", "mean"),
                max_dist_l2=("dist_l2", "max"),
                mean_dist_R=("dist_R", "mean"))
           .sort_values("mean_dist_l2", ascending=False))
display(summary.round(4))

print("\nStatus-Verteilung:")
display(lam_df["status"].value_counts().rename_axis("status").reset_index(name="n"))

unusable = lam_df[~lam_df["usable"]]
if len(unusable):
    print(f"\n{len(unusable)} Zeilen ohne lambda:")
    display(unusable[["p_name", "method", "params", "status"]])
    n_infeasible = int((unusable["status"] == "INFEASIBLE_BALL_MISSES_SIMPLEX").sum())
    n_failed = len(unusable) - n_infeasible
    print(f"  davon mathematisch leer: {n_infeasible}   Solver-Fehlschlaege: {n_failed}")
    if n_failed:
        print("  ACHTUNG: Solver-Fehlschlaege sind KEIN Ergebnis und muessen vor Phase B geklaert werden.")
else:
    print("\nAlle Zeilen mit verwertbarem lambda.")

### 6b. Konsistenz gegen NB09.1 und gegen die Theorie

Diese Zusicherungen sollen fehlschlagen, wenn die Mathematik bricht — nicht, wenn sich
eine Zahl aendert.

In [ ]:
# (1) Cert gibt p auf jeder Praeferenz, solange der Floor kollabiert ist.
cert_rows = lam_df[lam_df["method"] == "Cert"]
assert not cert_rows["moved"].any(), "Cert bewegt sich — widerspricht NB09.1 run1."
print(f"[OK] Cert gibt p auf allen {len(cert_rows)} Praeferenzen. Status: {sorted(cert_rows['status'].unique())}")

# (2) MaxMin(c>=1) kollabiert auf p (Prop 30(a)).
mm1 = lam_df[(lam_df["method"] == "MaxMin") & (lam_df["params"] == "c=1.0") & lam_df["usable"]]
if len(mm1):
    max_shift = float(np.nanmax(mm1["dist_l2"]))
    assert max_shift < 1e-6, f"MaxMin(c=1.0) bewegt sich (max {max_shift:.2e}) — Prop 30(a) verletzt."
    print(f"[OK] MaxMin(c=1.0) kollabiert auf p (max {max_shift:.2e}, {len(mm1)} Praeferenzen).")

# (3) MaxMin ist an Vertices fuer c<1 leer — kein Solver-Artefakt.
for pname, p in PREF_SET:
    if np.all(p > 1e-12):
        continue
    sub = lam_df[(lam_df["p_name"] == pname) & (lam_df["method"] == "MaxMin")]
    for _, r in sub.iterrows():
        c_value = float(r["params"].split("=")[1])
        if c_value < 1.0 - 1e-9:
            assert r["status"] == "INFEASIBLE_BALL_MISSES_SIMPLEX", \
                f"{pname}/c={c_value}: erwartet leer, bekommen {r['status']}"
print("[OK] MaxMin an Randpraeferenzen fuer c<1 durchgaengig als leer gemeldet.")

# (4) Jeder Mover verschlechtert mindestens eine Proxy-Achse (trivialer Floor).
movers = lam_df[lam_df["moved"] & lam_df["min_delta"].notna()]
violations = movers[movers["min_delta"] > 1e-9]
assert len(violations) == 0, f"{len(violations)} Mover ohne Verschlechterung — Floor waere nicht trivial."
print(f"[OK] Alle {len(movers)} bewegenden lambda verschlechtern mindestens eine Proxy-Achse.")

# (5) Fair ist praeferenz-blind, SOLANGE die Simplex-Schranken inaktiv sind.
#     Das Ziel U_alpha(R v + eps) haengt nur von v ab; p tritt allein ueber
#     -p_i <= v_i <= 1-p_i ein. Solange keine dieser Schranken bindet, ist die
#     Loesung fuer alle inneren p identisch. Bei alpha = 0 ist das Ziel linear, das
#     Optimum liegt also auf dem Rand des relaxierten Floors, und dort binden die
#     Schranken sehr wohl -- deshalb wird nur fuer alpha >= 1 zugesichert und fuer
#     alpha = 0 lediglich berichtet.
fair_interior = lam_df[(lam_df["method"] == "Fair") & lam_df["usable"] &
                       lam_df["p_name"].isin(INTERIOR)]
blind_rows = []
for params, group in fair_interior.groupby("params"):
    shifts = group[lam_cols].to_numpy(float) - group[p_cols].to_numpy(float)
    spread = float(np.abs(shifts - shifts[0]).max()) if len(shifts) > 1 else 0.0
    alpha_value = float(params.split(",")[0].split("=")[1])
    active = float(np.abs(shifts).max())
    blind_rows.append({"params": params, "alpha": alpha_value,
                       "spread_over_interior_p": spread,
                       "max_abs_shift": active,
                       "preference_blind": spread < 1e-5})
    if alpha_value >= 1.0 - 1e-9:
        assert spread < 1e-5, (
            f"Fair({params}): Verschiebung variiert um {spread:.2e} ueber innere p, "
            "obwohl bei alpha >= 1 eine innere Loesung erwartet wird."
        )
display(pd.DataFrame(blind_rows).round(6))
print("[OK] Fair ist fuer alpha >= 1 praeferenz-blind (innere Loesung, Schranken inaktiv).")
print("     Bei alpha = 0 ist das Ziel linear und das Optimum liegt auf dem Rand des")
print("     relaxierten Floors; dort binden die Simplex-Schranken, und p wirkt ueber sie.")
print("     In Kap. 3 gehoert beides hin -- die Blindheit UND ihre Grenze.")

# (6) Bei alpha = 0 wird die Relaxation voll ausgeschoepft: min_delta = -eps exakt.
#     Das ist Bang-Bang-Verhalten, kein Kompromiss, und bei grossem eps ein sehr
#     grosser Schritt. Als Methodeneinstellung ist es dokumentationsbeduerftig.
fair0 = lam_df[(lam_df["method"] == "Fair") & lam_df["params"].str.startswith("alpha=0.0") &
               lam_df["usable"]]
if len(fair0):
    for params, group in fair0.groupby("params"):
        eps_value = float(params.split("eps=")[1])
        worst = float(group["min_delta"].min())
        print(f"     Fair({params}): min_delta = {worst:.4f} gegen -eps = {-eps_value:.4f}, "
              f"max ||lambda-p|| = {group['dist_l2'].max():.4f}")

### 6c. Budget fuer Phase B

Ein Merge wird einmal ausgewertet, auch wenn mehrere Methoden dasselbe `lambda` liefern.
Cert und MaxMin(c>=1) fallen mit `lambda = p` auf die Baseline und kosten nichts extra.

### 6b+. Evaluationsprompts

Die Prompts entstehen im Notebook, damit `n_prompts` in der Vorregistrierung nicht von einem
separat ausgefuehrten Skript abhaengt. Gezogen wird aus dem VALIDATION-Split — den TRAIN-Split
haben die RS-PPO-Adapter in ihren PPO-Schritten gesehen. Prompttexte aus frueheren
Promptdateien werden ausgeschlossen, und die Zusammenfassung berichtet, wie viele es waren.


In [ ]:
from src.eval_prompts import build_eval_prompt_file

N_EVAL_PROMPTS = 80
EVAL_PROMPT_SEED = 137

# Fail-closed: diese Dateien MUESSEN gefunden werden, sonst bricht der Bau ab. Prompts zu
# ziehen, weil eine Ausschlussdatei nicht auffindbar war, ist genau der Fehler, den dieser
# Waechter verhindern soll. Fehlt eine laut Projektverlauf wirklich, bewusst
# ALLOW_MISSING_EXCLUSIONS = True setzen UND die Luecke berichten.
ALLOW_MISSING_EXCLUSIONS = False

REWARD_PROMPT_PATH.parent.mkdir(parents=True, exist_ok=True)
PROMPT_SUMMARY = build_eval_prompt_file(
    REWARD_PROMPT_PATH,
    n=N_EVAL_PROMPTS,
    seed=EVAL_PROMPT_SEED,
    project_root=PROJECT_ROOT,
    allow_missing_exclusions=ALLOW_MISSING_EXCLUSIONS,
)
print(json.dumps(PROMPT_SUMMARY, indent=2, ensure_ascii=False))
if not PROMPT_SUMMARY.get("disjointness_verified", False):
    print("\nHINWEIS: Disjunktheit zu frueheren Evaluationen ist NICHT nachgewiesen. "
          "So berichten, nicht anders.")


In [ ]:
eval_points, origins = [], {}

def _register(vec, origin):
    key = lambda_key(vec, decimals=LAMBDA_DEDUP_DECIMALS)
    if key not in origins:
        origins[key] = []
        eval_points.append(np.asarray(vec, dtype=np.float64))
    origins[key].append(origin)

# Nur die Phase-B-Teilmenge wird gemergt. Bei USE_FULL_SEARCH_SET = True bleibt Phase A
# vollstaendig (75 Punkte), waehrend Phase B bei den 11 benannten bezahlbar bleibt.
lam_df_B = lam_df[lam_df["p_name"].isin(PHASE_B_NAMES)]
assert len(lam_df_B) > 0, "Keine lambda-Zeile faellt in die Phase-B-Teilmenge."

for _, r in lam_df_B.iterrows():
    _register(r[p_cols].to_numpy(float), f"baseline:{r['p_name']}")
    if bool(r["usable"]) and bool(r["moved"]):
        _register(r[lam_cols].to_numpy(float), f"{r['method']}({r['params']}):{r['p_name']}")

EVAL_POINTS = np.asarray(eval_points, dtype=np.float64)
n_unique = len(EVAL_POINTS)

# Promptzahl aus der Promptdatei, nicht geraten.
if REWARD_PROMPT_PATH.is_file():
    n_prompts = sum(1 for line in REWARD_PROMPT_PATH.read_text(encoding="utf-8").splitlines() if line.strip())
else:
    n_prompts = 80
    print(f"HINWEIS: {REWARD_PROMPT_PATH.name} existiert noch nicht; rechne mit {n_prompts} Prompts.")

print(f"Eindeutige Merge-Punkte:   {n_unique}")
print(f"Prompts je Punkt:          {n_prompts}")
print(f"Generierungen gesamt:      {n_unique * n_prompts}")
print(f"Ohne Dedup waeren es:      {len(lam_df) * n_prompts}")
print(f"Ersparnis durch Dedup:     {100 * (1 - n_unique / max(len(lam_df), 1)):.1f} %")
hours = n_unique * n_prompts * 2.5 / 3600
print(f"\nGrobe Laufzeit (A100, 256 neue Token, ~2.5 s je Generierung + Scoring): ~{hours:.1f} h")
print("  " + ("PASST in eine Sitzung" if hours < 3 else
              "ZU LANG fuer eine Sitzung — Praeferenzmenge oder Gitter kuerzen"))

## 7. GATE — Vorregistrierung

**Ab hier wird es einmalig.** Diese Zelle friert fest, was gemessen wird, bevor irgendeine
Reward-Zahl gesehen wurde: Methoden, Gitter, Praeferenzmenge, Metrik, Prompts,
Entscheidungsregel. Der SHA256 ueber die lambda-Tabelle bindet die Vorregistrierung an genau
diese Punkte; aendert sich spaeter ein Gitter, passt der Hash nicht mehr und Phase B startet
nicht.

Vor dem Ausfuehren muessen geklaert sein: **AKUT #6** (SFT-Pipeline-Audit), die Provenienz von
`reward_matrix.npy`, und Lingxiaos Freigabe fuer Avg(H), MaxMin(c) und Fair(alpha, eps).

### 7-. Bindungsnachweis

Der lambda-Hash bindet nur die Koeffiziententabelle — also ausgerechnet den Teil, der
reward-frei jederzeit reproduzierbar ist. Prompts, Adaptergewichte, R-Matrizen, der
Scorer-Quelltext und das Generierungsmodell gehen bisher nirgends ein. Ohne sie koennte man
nach dem Lauf die Prompts austauschen, und der Hash passte weiter.


In [ ]:
def _sha256_file(path):
    """Return the SHA256 of one file."""
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def _sha256_dir(path, patterns=("*.safetensors", "*.bin", "adapter_config.json")):
    """Return a stable SHA256 over the relevant files of one adapter directory."""
    digest = hashlib.sha256()
    for pattern in patterns:
        for file_path in sorted(Path(path).glob(pattern)):
            digest.update(file_path.name.encode())
            digest.update(_sha256_file(file_path).encode())
    return digest.hexdigest()

def _model_revisions():
    """Best-effort exact revisions of the two external models."""
    revisions = {}
    for label, repo in (("base_model", str(config["base_model_name"])),
                        ("armorm", "RLHFlow/ArmoRM-Llama3-8B-v0.1")):
        try:
            from huggingface_hub import HfApi

            revisions[label] = HfApi().model_info(repo).sha
        except Exception as error:                      # offline or no hub access
            revisions[label] = f"unresolved: {type(error).__name__}"
    return revisions

BINDING = {
    "regime": REGIME,
    "adapter_pattern": ADAPTER_PATTERN,
    "adapters_sha256": {a: _sha256_dir(p) for a, p in ADAPTER_PATHS.items()},
    "matrix_cos_sha256": _sha256_file(COSINE_MATRIX_CSV),
    "matrix_gram_sha256": _sha256_file(GRAM_MATRIX_CSV),
    "prompts_sha256": _sha256_file(REWARD_PROMPT_PATH),
    "scorer_sha256": _sha256_file(PROJECT_ROOT / "src" / "armorm_scorer.py"),
    "portfolio_sha256": _sha256_file(PROJECT_ROOT / "src" / "coefficient_portfolio.py"),
    "base_model_name": str(config["base_model_name"]),
    "armorm_precision": ARMORM_PRECISION,
    "attribute_order": list(ATTRIBUTES),
    "generation": {"max_new_tokens": MAX_NEW_TOKENS,
                   "repetition_penalty": REPETITION_PENALTY,
                   "no_repeat_ngram_size": NO_REPEAT_NGRAM_SIZE,
                   "decoding": "greedy, do_sample=False, num_beams=1"},
    "grids": {"rho_avg": RHO_AVG, "cert_c": CERT_C, "cert_eps": CERT_EPS,
              "c_grid": list(C_GRID), "alpha_grid": list(ALPHA_GRID), "eps_grid": list(EPS_GRID)},
    "src_sha256": {name: _sha256_file(PROJECT_ROOT / "src" / name) for name in (
        "merge.py", "proxy_validation.py", "metrics.py", "lambda_utils.py",
        "armorm_objectives.py", "eval_prompts.py")},
    "model_revisions": _model_revisions(),
}

# Ein einziger Hash ueber den gesamten Nachweis. Er wandert in den Reward-Cache, damit
# eine unterbrochene Sitzung nicht unter geaenderten Bedingungen fortgesetzt wird.
BINDING_SHA256 = hashlib.sha256(
    json.dumps(BINDING, sort_keys=True).encode()).hexdigest()
print(json.dumps(BINDING, indent=2))
print()
print(f"BINDING_SHA256 = {BINDING_SHA256}")


In [ ]:
PREREG_CONFIRM = False   # bewusst auf True setzen, wenn die Vorregistrierung stehen soll

lambda_hash = hashlib.sha256(
    lam_df[["p_name", "method", "params", "status"] + lam_cols]
    .round(9).to_csv(index=False).encode()
).hexdigest()

prereg = {
    "run_tag": RUN_TAG,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "NB10 method comparison",
    "lambda_table_sha256": lambda_hash,
    "primary_matrix": PRIMARY_MATRIX,
    "methods": {
        "Avg": {"source": "src.coefficient_portfolio.m1_plus", "rho": RHO_AVG},
        "Cert": {"source": "src.coefficient_portfolio.c1_plus_plus", "c": CERT_C, "eps": CERT_EPS,
                 "note": "certificate row: lambda = p by NB09.1 run1 (t*=0 for 77/77, "
                         "R^-1 1 > 0); not measured separately"},
        "MaxMin": {"source": "src.coefficient_portfolio.maxmin_c", "c_grid": C_GRID,
                   "note": "empty at simplex vertices for every c < 1; reported as infeasible, not as collapse"},
        "Fair": {"source": "src.coefficient_portfolio.fair_alpha_eps",
                 "alpha_grid": ALPHA_GRID, "eps_grid": EPS_GRID,
                 "note": "objective depends on the displacement only; p enters through the "
                         "simplex bounds. NOT preference-aware in the sense of RQ1."},
    },
    "baseline": "lambda = p",
    "n_preferences_phase_a": len(PREF_SET),
    "n_preferences_phase_b": len(PREF_SET_B),
    "phase_b_preference_rule": PHASE_B_RULE,
    "preference_names": [n for n, _ in PREF_SET],
    "phase_b_preference_names": [n for n, _ in PREF_SET_B],
    "preference_set_note": (
        "11 named preferences. Deviates from the 13 pre-registered in NB09.1: "
        "quality_focused and detailed_answer removed, uniform renamed to balanced. "
        "Report this deviation explicitly."
    ),
    "binding": BINDING,
    "binding_sha256": BINDING_SHA256,
    "prompt_provenance": PROMPT_SUMMARY,
    "reward_model": {
        "name": "RLHFlow/ArmoRM-Llama3-8B-v0.1",
        "precision": ARMORM_PRECISION,
        "batch_size": 1,
        "head_mapping": "src.armorm_objectives.helpsteer_head_indices (external golden sample)",
        "scoring_format": "apply_chat_template(user + assistant)",
    },
    "generation_format": "apply_chat_template(user, add_generation_prompt=True) - identical to NB08",
    "raw_scores_retained": True,
    "error_layer": "paired bootstrap over prompts, 10000 draws, alpha=0.05",
    "multiplicity": "Holm-Bonferroni over all (method, preference) comparisons",
    "n_unique_merge_points": int(n_unique),
    "metric_primary": "raw preference-weighted sum U_p = sum_i p_i r_i; identity normalization r~_i := r_i",
    "metric_implementation": "src.proxy_validation.preference_utility(..., normalization='identity')",
    "metric_robustness": ["minmax", "rank"],
    "metric_note": "supersedes rank-U_p from NB06.1 metric set v2",
    "prompts": {"path": str(REWARD_PROMPT_PATH.relative_to(PROJECT_ROOT)),
                "n": int(n_prompts),
                "max_new_tokens": MAX_NEW_TOKENS,
                "repetition_penalty": REPETITION_PENALTY,
                "no_repeat_ngram_size": NO_REPEAT_NGRAM_SIZE,
                "decoding": "greedy, do_sample=False, num_beams=1"},
    "decision_rule": (
        "Primary: sign and magnitude of Delta U_p = U_p(lambda) - U_p(p) per method. "
        "Confirmatory improvement requires Delta U_p > 0 and a Holm-adjusted p-value < 0.05 "
        "across all method/preference comparisons; harm is defined analogously for Delta U_p "
        "< 0. The unadjusted percentile bootstrap CI is descriptive only. No threshold is "
        "lowered after seeing the numbers. Cert is reported as a certificate row with Delta "
        "U_p = 0 by construction. Infeasible MaxMin cells are reported as infeasible and are "
        "not counted as collapses."
    ),
    "armorm_role": (
        "REGIME-DEPENDENT. In the RS-PPO regime ArmoRM is frozen during evaluation but "
        "was ALSO the PPO reward model, so this evaluation is circular and supports only "
        "upper-bound and diagnostic claims, not held-out proxy validity. The read-only "
        "firewall claim holds for the SFT regime only, and there only pending the AKUT #6 "
        "audit. Do not restate the firewall as a global property of the thesis."
    ),
}

if PREREG_JSON.exists():
    frozen = json.loads(PREREG_JSON.read_text(encoding="utf-8"))
    print(f"Vorregistrierung existiert bereits ({frozen['created_utc']}) — nicht ueberschrieben.")
    if frozen["lambda_table_sha256"] != lambda_hash:
        print("\n*** WARNUNG: lambda-Tabelle weicht von der eingefrorenen Fassung ab. ***")
        print(f"    eingefroren: {frozen['lambda_table_sha256'][:16]}")
        print(f"    aktuell:     {lambda_hash[:16]}")
        print("    Phase B bricht ab. Gitter zuruecksetzen oder neuen RUN_TAG waehlen.")
elif PREREG_CONFIRM:
    write_json(PREREG_JSON, prereg)
    print(f"Vorregistrierung geschrieben -> {PREREG_JSON}")
    print(f"lambda-Hash: {lambda_hash}")
else:
    print("PREREG_CONFIRM ist False — nichts geschrieben. Vorschau:")
    print(json.dumps(prereg, indent=2)[:2000])

In [ ]:
GATE_OPEN = False
if PREREG_JSON.exists():
    frozen = json.loads(PREREG_JSON.read_text(encoding="utf-8"))
    _diffs = []
    if frozen.get("lambda_table_sha256") != lambda_hash:
        _diffs.append("lambda_table_sha256")
    for _k, _v in BINDING.items():
        if frozen.get("binding", {}).get(_k) != _v:
            _diffs.append(f"binding.{_k}")
    GATE_OPEN = not _diffs
    if GATE_OPEN:
        print("GATE OFFEN — Vorregistrierung passt zu lambda-Tabelle UND Bindungsnachweis.")
    else:
        print("GATE ZU — Abweichung in: " + ", ".join(_diffs))
        print("  Etwas hat sich seit dem Einfrieren geaendert: Prompts, Adapter, Matrix, "
              "Scorer oder Gitter. Ursache klaeren, nicht neu einfrieren.")
else:
    print("GATE ZU — keine Vorregistrierung.")

if RUN_REWARD_COLLECTION and not GATE_OPEN:
    raise RuntimeError(
        "RUN_REWARD_COLLECTION ist True, aber das Gate ist zu. Ohne eingefrorene, "
        "passende Vorregistrierung wird ArmoRM nicht angefasst."
    )

## 8. Phase B — Reward-Auswertung

Merge-Arithmetik aus `src/merge.py`, Cache und Wiederaufnahme aus
`src.proxy_validation.collect_reward_tensor`. Das Notebook legt nur den Kontextmanager
darum, der die Gewichte nach jedem Punkt wiederherstellt — `apply_effective_deltas_to_model`
ist destruktiv, und das Modell 51-mal neu zu laden waere unbezahlbar.

In [ ]:
if RUN_REWARD_COLLECTION:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    from src.merge import combine_effective_deltas, effective_deltas, resolve_base_module

    BASE_MODEL_NAME = str(config["base_model_name"])
    # Aufgeloest und gegen das Regime geprueft in Abschnitt 5a+. Hier NICHT erneut aus der
    # Config raten: config["adapter_dir"] zeigt auf die SFT-Adapter.
    adapter_paths = dict(ADAPTER_PATHS)
    assert BINDING["adapters_sha256"] == {a: _sha256_dir(p) for a, p in adapter_paths.items()}, \
        "Adaptergewichte haben sich seit dem Bindungsnachweis geaendert."

    generation_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME, torch_dtype=torch.float32, device_map="auto")
    base_model.eval()
    print("Basismodell in float32 geladen (bf16 zerstoert die Merge-Endpunkte).")

    DELTAS = effective_deltas(adapter_paths)   # fp32 auf CPU, einmalig
    print(f"Effektive Deltas fuer {len(next(iter(DELTAS.values())))} Module vorbereitet.")
else:
    print("RUN_REWARD_COLLECTION ist False — Phase B uebersprungen.")

In [ ]:
if RUN_REWARD_COLLECTION:
    @contextmanager
    def merged_model(model, deltas_by_adapter, lam):
        """Apply sum_i lambda_i delta_i, then restore the original weights."""
        merged = combine_effective_deltas(lam, deltas_by_adapter)
        originals = {}
        try:
            for module_name, delta_cpu in merged.items():
                module = resolve_base_module(model, module_name)
                originals[module_name] = module.weight.detach().clone()
                update = (module.weight.detach().to(dtype=torch.float32)
                          + delta_cpu.to(device=module.weight.device, dtype=torch.float32))
                with torch.no_grad():
                    module.weight.copy_(update.to(dtype=module.weight.dtype))
            yield model
        finally:
            for module_name, original in originals.items():
                with torch.no_grad():
                    resolve_base_module(model, module_name).weight.copy_(original)
            del originals, merged
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    def generate_answer(prompt: str) -> str:
        device = next(base_model.parameters()).device
        # NB08-Format. Die RS-PPO-Adapter wurden mit apply_chat_template und
        # add_generation_prompt=True angesprochen; ein anderes Format hiesse,
        # off-distribution zu evaluieren. add_special_tokens=False, weil das Template
        # das Praefix bereits enthaelt — Kontrolle unten in dieser Zelle.
        text = generation_tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
        )
        encoded = generation_tokenizer(text, return_tensors="pt", add_special_tokens=False)
        input_ids = encoded["input_ids"].to(device)
        with torch.inference_mode():
            generated = base_model.generate(
                input_ids=input_ids,
                attention_mask=encoded["attention_mask"].to(device),
                max_new_tokens=MAX_NEW_TOKENS, do_sample=False, num_beams=1,
                repetition_penalty=REPETITION_PENALTY,
                no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
                pad_token_id=generation_tokenizer.eos_token_id)
        return generation_tokenizer.decode(generated[0, input_ids.shape[1]:], skip_special_tokens=True)

    _probe = generation_tokenizer.apply_chat_template(
        [{"role": "user", "content": "probe"}], tokenize=False, add_generation_prompt=True)
    _bos = generation_tokenizer.bos_token_id
    _ids = generation_tokenizer(_probe, add_special_tokens=False)["input_ids"]
    assert _bos is None or _ids.count(_bos) <= 1, "Doppeltes BOS im Generierungsprompt."
    print("Generierungsformat (NB08-identisch):")
    print(repr(_probe))
    print("Merge- und Generierungsroutinen bereit.")

In [ ]:
if RUN_REWARD_COLLECTION:
    from src.armorm_objectives import ARMORM_HELPSTEER_OBJECTIVE_NAMES
    from src.armorm_scorer import make_score_prompt_answer
    from src.proxy_validation import collect_reward_tensor
    from src.tinyllama_training_utils import load_reward_prompts

    reward_prompts = load_reward_prompts(REWARD_PROMPT_PATH)
    assert len(reward_prompts) == n_prompts, "Promptzahl weicht von der Vorregistrierung ab."
    assert all(a in ARMORM_HELPSTEER_OBJECTIVE_NAMES for a in ATTRIBUTES), \
        "ATTRIBUTES enthaelt eine Achse ohne verankerte ArmoRM-Kopfzuordnung."

    # 8-bit: identisch zum Belohnungspfad des PPO-Laufs. Der Golden Sample laeuft dadurch
    # bei der lockeren Toleranz 0.35 des Model Cards, weil int8 die Regressionskoepfe
    # leicht verschiebt.
    score_prompt_answer, scorer = make_score_prompt_answer(
        dtype="bfloat16", load_in_8bit=(ARMORM_PRECISION == "8bit"))
    assert scorer.describe()["precision"] == ("int8" if ARMORM_PRECISION == "8bit" else "bfloat16")
    scorer.assert_golden_sample()          # VOR dem ersten echten Scoring
    print("Scorer:", scorer.describe())

    def reward_of_lambda(lam):
        """Per-prompt ArmoRM rewards for one merge point, shape (n_prompts, m)."""
        with merged_model(base_model, DELTAS, lam):
            answers = [generate_answer(record["prompt"]) for record in reward_prompts]
        scores = np.asarray([
            score_prompt_answer(record["prompt"], answer, ATTRIBUTES)
            for record, answer in zip(reward_prompts, answers)
        ], dtype=np.float64)
        assert scores.shape == (len(reward_prompts), m), scores.shape
        return scores          # NICHT mitteln: der gepaarte Bootstrap braucht die Rohwerte

    REWARD_TENSOR = collect_reward_tensor(
        EVAL_POINTS, reward_of_lambda, REWARD_CACHE,
        num_prompts=n_prompts, binding_sha256=BINDING_SHA256)
    REWARD_MATRIX = REWARD_TENSOR.mean(axis=1)     # identisch zur frueheren Matrix
    np.save(RESULTS_DIR / f"nb10_{RUN_TAG}_reward_tensor.npy", REWARD_TENSOR)
    print(f"Reward-Tensor: {REWARD_TENSOR.shape}   Reward-Matrix: {REWARD_MATRIX.shape}")


## 9. Endtabelle

| Spalte | Bedeutung |
|---|---|
| `p_name`, `p` | Praeferenzvektor |
| `method`, `params` | Methode und Parametrisierung |
| `lambda` | gefundenes Mischungsverhaeltnis, `None` bei leerer zulaessiger Menge |
| `U_p(p)` | Baseline |
| `U_p(lambda)` | dieselbe Nutzenfunktion an `lambda` |
| `Delta U_p` | Differenz; positiv = besser als die Baseline |
| `dist_l2`, `dist_R` | Abweichung von `p`, euklidisch und in der `R`-Metrik |

`dist_R` steht daneben, weil die Theorie durchgaengig in der `R`-Norm formuliert ist —
Prop 20 lautet `Delta U_p = 2 rho b ||lambda_Avg - p||_R^2`. Wer nur `dist_l2` berichtet, kann
diese Identitaet nicht pruefen.

In [ ]:
def build_final_table() -> pd.DataFrame:
    have_rewards = REWARD_CACHE.is_file() and REWARD_CACHE.stat().st_size > 0
    reward_by_key = {}
    if have_rewards:
        for line in REWARD_CACHE.read_text(encoding="utf-8").splitlines():
            if line.strip():
                record = json.loads(line)
                reward_by_key[str(record["key"])] = np.asarray(record["reward"], dtype=np.float64)

    out = []
    for _, r in lam_df.iterrows():
        p = r[p_cols].to_numpy(float)
        lam = r[lam_cols].to_numpy(float)
        usable = bool(r["usable"])
        record = {
            "p_name": r["p_name"], "p": np.round(p, 4).tolist(),
            "method": r["method"], "params": r["params"], "status": r["status"],
            "lambda": np.round(lam, 4).tolist() if usable else None,
            "dist_l2": r["dist_l2"], "dist_R": r["dist_R"],
        }
        if have_rewards:
            r_p = reward_by_key.get(coefficient_key(p))
            u_base = float(preference_utility(r_p[None, :], p, "identity")[0]) if r_p is not None else np.nan
            if r["method"] == "Cert":
                u_lam, note = u_base, "certificate: lambda = p, Delta = 0 by construction"
            elif not usable:
                u_lam, note = np.nan, r["status"]
            else:
                r_l = reward_by_key.get(coefficient_key(lam))
                u_lam = float(preference_utility(r_l[None, :], p, "identity")[0]) if r_l is not None else np.nan
                note = "" if r_l is not None else "reward missing"
            record.update({"U_p(p)": u_base, "U_p(lambda)": u_lam,
                           "Delta U_p": u_lam - u_base, "note": note})
        else:
            record.update({"U_p(p)": np.nan, "U_p(lambda)": np.nan, "Delta U_p": np.nan,
                           "proxy_pRlam": r["proxy_pRlam"],
                           "note": "Phase B nicht gelaufen"})
        out.append(record)
    return pd.DataFrame(out)


final_df = build_final_table()
final_df.to_csv(FINAL_CSV, index=False)
print(f"Endtabelle -> {FINAL_CSV}  ({len(final_df)} Zeilen)")
with pd.option_context("display.max_rows", 250, "display.width", 240):
    display(final_df.round(5))

### 9a. Verdichtete Sicht je Methode

Ohne Phase B bleiben die `U_p`-Spalten leer. Das ist Absicht: eine Proxy-Zahl an die Stelle
der Reward-Zahl zu setzen waere genau die Verwechslung, gegen die Wand A argumentiert.

In [ ]:
agg = {"n": ("dist_l2", "size"),
       "n_usable": ("lambda", lambda s: int(s.notna().sum())),
       "mean_dist_l2": ("dist_l2", "mean"),
       "mean_dist_R": ("dist_R", "mean")}
if final_df["Delta U_p"].notna().any():
    agg.update({"mean_Delta_U_p": ("Delta U_p", "mean"),
                "n_improved": ("Delta U_p", lambda s: int((s > 0).sum())),
                "n_worse": ("Delta U_p", lambda s: int((s < 0).sum()))})
grouped = final_df.assign(
    key=final_df["method"] + final_df["params"].map(lambda s: f"({s})" if s else "")
).groupby("key").agg(**agg)
display(grouped.round(5))

### 9b. Robustheit der Normalisierung

Der Einwand gegen die rohe Summe lautet, dass die Achsen auf nativen Skalen eingehen und
sich um etwa den Faktor drei unterscheiden. Die Antwort ist keine Annahme, sondern eine
Rechnung: dieselbe Reward-Matrix, drei Normalisierungen, ein Vergleich der induzierten
Rangfolgen. Kostet null zusaetzliche Auswertungen.

In [ ]:
if REWARD_CACHE.is_file() and REWARD_CACHE.stat().st_size > 0:
    matrix = np.asarray([
        json.loads(line)["reward"]
        for line in REWARD_CACHE.read_text(encoding="utf-8").splitlines() if line.strip()
    ], dtype=np.float64)

    robust_rows = []
    for pname, p in PREF_SET:
        agreement = normalization_agreement(matrix, p)
        robust_rows.append({
            "p_name": pname,
            "argmax_agrees": agreement["argmax_agrees"],
            **{f"argmax_{k}": v for k, v in agreement["argmax_index"].items()},
            **{f"rho_{k}": v for k, v in agreement["spearman"].items()},
        })
    robust_df = pd.DataFrame(robust_rows)
    robust_df.to_csv(ROBUST_CSV, index=False)
    display(robust_df.round(4))
    n_agree = int(robust_df["argmax_agrees"].sum())
    print(f"\nargmax stimmt bei {n_agree}/{len(robust_df)} Praeferenzen ueber alle drei Normalisierungen ueberein.")
    print("Stimmt er durchgaengig, ist der Skalen-Einwand gegen die rohe Summe empirisch beantwortet;")
    print("weicht er ab, muss das als Einschraenkung berichtet werden — nicht als Anlass, die Metrik zu wechseln.")
else:
    print("Kein Reward-Cache — Robustheitsvergleich uebersprungen.")

### 9c. Fehlerschicht und Multiplizitaet

`Delta U_p > 0` allein ist kein Ergebnis. Der gepaarte Bootstrap laeuft ueber dieselben
Prompts fuer `lambda` und fuer `p`, sodass die Promptschwierigkeit herausfaellt. Bei rund
zweihundert Vergleichen erzeugt `alpha = 0.05` etwa zehn Zufallstreffer, deshalb Holm ueber
alle Paare (Methode, Praeferenz). Diese Zelle steht VOR dem Export, damit `stats.csv` im ZIP
landet.


In [ ]:
STATS_CSV = RESULTS_DIR / f"nb10_{RUN_TAG}_stats.csv"

if RUN_REWARD_COLLECTION:
    from src.lambda_utils import holm_adjust, lambda_key
    from src.metrics import mean_rank, paired_bootstrap_ci, selection_regret

    # Gleiche Schluesselfunktion wie die Deduplikation in Abschnitt 6c, sonst findet
    # der Lookup die Punkte nicht wieder.
    def _key(vec):
        return lambda_key(np.asarray(vec, dtype=float), decimals=LAMBDA_DEDUP_DECIMALS)

    point_index = {_key(pt): i for i, pt in enumerate(EVAL_POINTS)}
    stats_rows, utilities_by_method = [], {}

    for _, r in lam_df_B.iterrows():
        if not (bool(r["usable"]) and bool(r["moved"])):
            continue
        p_vec = r[p_cols].to_numpy(float)
        lam_vec = r[lam_cols].to_numpy(float)
        i_lam, i_p = point_index.get(_key(lam_vec)), point_index.get(_key(p_vec))
        if i_lam is None or i_p is None:
            raise KeyError(f"Merge-Punkt fehlt im Tensor: {r['p_name']}/{r['method']}")

        boot = paired_bootstrap_ci(REWARD_TENSOR[i_lam], REWARD_TENSOR[i_p], p_vec)
        label = f"{r['method']}({r['params']})" if r["params"] else str(r["method"])
        stats_rows.append({
            "p_name": r["p_name"], "method": r["method"], "params": r["params"], "label": label,
            "U_p_baseline": float(REWARD_TENSOR[i_p].mean(axis=0) @ p_vec),
            "U_p_lambda": float(REWARD_TENSOR[i_lam].mean(axis=0) @ p_vec),
            **{k: boot[k] for k in ("delta_u_p", "ci_low", "ci_high", "excludes_zero", "p_value")},
        })
        utilities_by_method.setdefault(label, {})[r["p_name"]] = stats_rows[-1]["U_p_lambda"]
        utilities_by_method.setdefault("Baseline (lambda=p)", {})[r["p_name"]] = \
            stats_rows[-1]["U_p_baseline"]

    stats_df = pd.DataFrame(stats_rows)
    if len(stats_df):
        # Konfirmatorisch zaehlt AUSSCHLIESSLICH Holm. Das unadjustierte CI bleibt als
        # deskriptive Spalte erhalten, traegt aber keine Aussage: bei rund zweihundert
        # Vergleichen erzeugt alpha = 0.05 etwa zehn Zufallstreffer.
        stats_df["p_holm"] = holm_adjust(stats_df["p_value"].to_numpy(float))
        stats_df["holm_improves"] = (stats_df["delta_u_p"] > 0) & (stats_df["p_holm"] < 0.05)
        stats_df["holm_harms"] = (stats_df["delta_u_p"] < 0) & (stats_df["p_holm"] < 0.05)
        stats_df["significant"] = stats_df["holm_improves"] | stats_df["holm_harms"]

        common = set.intersection(*(set(v) for v in utilities_by_method.values()))
        if common:
            order = sorted(common)
            ranks = mean_rank({k: [v[n] for n in order] for k, v in utilities_by_method.items()})
            print(f"Mean Rank ueber {len(order)} Praeferenzen (1 = bester):")
            for name, value in sorted(ranks.items(), key=lambda kv: kv[1]):
                print(f"  {value:5.2f}  {name}")

        # Selection Regret gegen den besten Punkt der GESAMTEN evaluierten Menge, nicht
        # nur gegen die fuer diese Praeferenz erzeugten. Nur so ist der Referenzsatz fuer
        # alle Methoden identisch — und nur so deckt sich die Zahl mit der Formulierung
        # "bester evaluierter Punkt" in der Arbeit.
        for pname, p_vec in PREF_SET_B:
            search_utilities = (REWARD_MATRIX @ np.asarray(p_vec, dtype=float)).tolist()
            mask = stats_df["p_name"] == pname
            stats_df.loc[mask, "selection_regret"] = [
                selection_regret(u, search_utilities) for u in stats_df.loc[mask, "U_p_lambda"]]
            stats_df.loc[mask, "baseline_regret"] = selection_regret(
                float(stats_df.loc[mask, "U_p_baseline"].iloc[0]), search_utilities)

        stats_df.to_csv(STATS_CSV, index=False)
        print(f"\n{len(stats_df)} Vergleiche, "
              f"{int(stats_df['excludes_zero'].sum())} mit CI ohne Null, "
              f"{int(stats_df['holm_improves'].sum())} nach Holm besser, "
              f"{int(stats_df['holm_harms'].sum())} nach Holm schlechter.")
        display(stats_df.sort_values("delta_u_p", ascending=False).round(5))
    else:
        print("Keine bewegenden lambda in der Phase-B-Teilmenge: unter Floor-Kollaps faellt "
              "jede Methode mit lambda = p auf die Baseline. Das ist das Zertifikat aus "
              "NB09.1, kein fehlendes Ergebnis.")
else:
    print("RUN_REWARD_COLLECTION ist False — Statistik uebersprungen.")


## 10. Export

In [ ]:
report = {
    "run_tag": RUN_TAG,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "NB10 method comparison",
    "primary_matrix": PRIMARY_MATRIX,
    "methods_from": "src.coefficient_portfolio (no method defined in this notebook)",
    "n_preferences_phase_a": len(PREF_SET),
    "n_preferences_phase_b": len(PREF_SET_B),
    "regime": REGIME,
    "armorm_precision": ARMORM_PRECISION,
    "n_lambda_rows": int(len(lam_df)),
    "n_unique_merge_points": int(n_unique),
    "phase_b_run": bool(RUN_REWARD_COLLECTION),
    "gate_open": bool(GATE_OPEN),
    "lambda_table_sha256": lambda_hash,
    "metric_primary": "raw U_p, identity normalization",
    "cert_note": "certificate row from NB09.1 run1; not measured",
    "status_counts": lam_df["status"].value_counts().to_dict(),
}
write_json(REPORT_JSON, report)

zip_path = RESULTS_DIR / f"nb10_{RUN_TAG}_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in (LAMBDA_CSV, FINAL_CSV, ROBUST_CSV, STATS_CSV, REPORT_JSON, PREREG_JSON,
                 REWARD_CACHE, REWARD_PROMPT_PATH):
        if path.is_file():
            archive.write(path, path.name)
print(f"{zip_path}\nSHA256 {hashlib.sha256(zip_path.read_bytes()).hexdigest()}")

try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    pass

In [ ]:
!git status --short

## 11. Was noch offen ist

**Vor dem Gate zu klaeren, nicht im Notebook loesbar**

* **AKUT #6 — SFT-Pipeline-Audit.** Entscheidet, ob RQ2 regime-beschraenkt gilt oder
  entfaellt. Zu pruefen ist, dass ArmoRM im SFT-Pfad ausschliesslich der finalen Evaluation
  diente: nicht der Checkpoint-Auswahl, nicht dem Early Stopping, nicht der
  Hyperparameterwahl, nicht der Datenfilterung, nicht der Kandidatenauswahl, nicht dem
  Equal-N-Schnitt.
* **Provenienz von `reward_matrix.npy`** — SFT- oder PPO-Bundle.
* **Abweichung der Praeferenzmenge.** NB09.1 registrierte 13 vor; hier laufen 11.
  `quality_focused` und `detailed_answer` sind entfernt, `uniform` heisst `balanced`. Das
  gehoert als Abweichung berichtet, nicht stillschweigend uebernommen.

**Vor dem Gate im Notebook zu entscheiden**

* **Budget.** Die Schaetzung in Abschnitt 6c rechnet mit 2.5 s je Punkt. An fuenf Punkten
  nachmessen, bevor eingefroren wird. Reicht die Zeit nicht, ist Fair(alpha=0) der
  schmerzfreieste Schnitt: das Ziel ist dort linear, `min_delta` trifft exakt `-eps`, und der
  Schritt schreibt `lambda` neu, statt es zu korrigieren. Kuerzen ist NACH dem Gate nicht
  mehr zulaessig.

**Erledigt**

* Scorer: `src/armorm_scorer.py`, Kopf-Aufloesung ueber `helpsteer_head_indices`, Golden
  Sample vor dem ersten echten Scoring, `batch_size=1`, ArmoRM in 8-bit wie im PPO-Lauf.
* Rohwerte je Prompt bleiben erhalten (`collect_reward_tensor`), Fehlerschicht ist der
  gepaarte Bootstrap mit Holm-Korrektur.
* `RHO_AVG` fest, Praeferenzmenge fest, Generierung im NB08-Format.
* Bindungsnachweis ueber Prompts, Adapter, Matrizen, Scorer und Basismodell; das Gate prueft
  ihn vollstaendig.

**Bleibt als Hygiene ausserhalb dieses Notebooks**

* `ARMORM_HELPSTEER_OBJECTIVES` in `tinyllama_training_utils.py` hartcodiert die Indizes 0-4;
  ebenso ein Dict in NB06.1 Zelle 27. Nur `armorm_objectives.py` ist per Golden Sample
  verankert, die anderen sollten darauf umgestellt werden.
* `src/coefficient_methods.py` ist ein drittes, totes Portfolio (Prae-v13-Nomenklatur) und
  genau das Muster, nach dem Gate G6 sucht.
